# Failure Analysis

## Learning goals

- understand common failure modes in agent systems
- inspect the reusable failure taxonomy and severity metadata
- connect failure summaries to traces and concrete example runs
- turn observed failures into prioritized improvement actions


## Concept explanation

As with the other notebooks, we begin by confirming the runtime. Failure analysis only helps if you know which environment produced the artifacts you are studying.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## Failure modes in agent systems

Agent systems fail in more than one place. Retrieval can miss evidence, planners can choose shallow decompositions, synthesis can omit important details, and fallback policies can answer when they should abstain or abstain when they should answer. A taxonomy makes those patterns discussable.


## Implementation

This notebook imports the reusable taxonomy and analyzer modules rather than redefining failure logic inline. That keeps the debugging workflow aligned with the production evaluation code.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.config import get_paths
from src.evaluator import attach_failure_improvements, extract_failure_cases, run_evaluation_suite
from src.failure_analyzer import analyze_failures, generate_failure_report, taxonomy_frame
from src.trace_debug import display_trace
from src.utils import read_json

paths = get_paths()
results_path = paths.eval_dir / 'eval_results.json'
if results_path.exists():
    results = pd.DataFrame(read_json(results_path))
else:
    results, _ = run_evaluation_suite(repeats=2, persist_outputs=True)

results.head(5)

## Failure taxonomy

The taxonomy below makes each failure more operational. Stage tells us where to look, severity tells us how urgent it is, and mitigation gives us a first improvement hypothesis.


In [ ]:
taxonomy = taxonomy_frame().sort_values(['severity', 'stage', 'failure_type']).reset_index(drop=True)
severity_colors = {'critical': '#FEE2E2', 'major': '#FEF3C7', 'minor': '#DBEAFE'}

def color_row(row):
    color = severity_colors.get(row['severity'], '#FFFFFF')
    return [f'background-color: {color}' for _ in row]

taxonomy.style.apply(color_row, axis=1)

## Extract failure cases

Now we pull only the failing runs and attach improvement hints. This gives us a working table for the rest of the notebook and mirrors how a real research loop narrows from all runs to only the problematic ones.


In [ ]:
failures = attach_failure_improvements(extract_failure_cases(results))
failures[['system', 'question_id', 'expected_question_type', 'failure_type', 'improvement_idea']].head(12)

## Trace inspection

Aggregate counts are useful, but single traces explain mechanism. The next cell inspects the first few failure traces recorded for the agent workflow and makes it easy to see which node introduced the problem.


In [ ]:
from pathlib import Path

analysis = analyze_failures(failures)
report_path = paths.reports_dir / 'failure_report.md'
generate_failure_report(analysis, report_path)

agent_failures = failures[failures['system'] == 'agent_workflow'].head(3)
trace_previews = []
for _, row in agent_failures.iterrows():
    trace_path = paths.traces_dir / f"{row['question_id']}_run{int(row['run_id'])}.json"
    trace = read_json(trace_path)['trace'] if trace_path.exists() else []
    trace_previews.append(
        {
            'question_id': row['question_id'],
            'failure_type': row['failure_type'],
            'trace_path': str(trace_path),
            'trace_steps': len(trace),
        }
    )

trace_preview_frame = pd.DataFrame(trace_previews)
display(trace_preview_frame)
for preview in trace_previews:
    print(f"Trace for {preview['question_id']} ({preview['failure_type']})")
    display_trace(read_json(Path(preview['trace_path']))['trace'])

## Experiment

Once failures are labeled, we can ask where they cluster. Stage and severity views help decide whether the next iteration should focus on retrieval, planning, synthesis, or fallback policy.


In [ ]:
stage_distribution = pd.Series(analysis['stage_distribution']).sort_values(ascending=False)
severity_distribution = pd.Series(analysis['severity_distribution']).sort_values(ascending=False)
improvement_actions = pd.DataFrame(analysis['top_improvement_actions'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
stage_distribution.plot(kind='bar', color='#4C78A8', ax=axes[0], title='Failures by Stage')
axes[0].set_xlabel('stage')
axes[0].set_ylabel('count')
severity_distribution.plot(kind='pie', autopct='%1.0f%%', ax=axes[1], title='Failure Severity Mix')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

display(improvement_actions)

## Result analysis

A failure taxonomy is useful because it shortens the path from symptom to intervention. Instead of saying "the workflow seems weak," we can now say "most errors come from retrieval noise" or "critical issues cluster in fallback logic," which leads directly to targeted fixes.


In [ ]:
pd.Series({
    'total_failures': int(analysis['total_failure_instances']),
    'unique_failure_types': len(analysis['failure_distribution']),
    'report_path': str(report_path),
    'top_improvement_action': improvement_actions.iloc[0]['mitigation'] if not improvement_actions.empty else 'None',
})

## Takeaways

- Failure analysis turns bad runs into structured learning assets.
- Stage and severity metadata help prioritize work instead of reacting to anecdotes.
- Trace inspection connects high-level labels to specific workflow decisions.
- A research-grade agent project should always pair evaluation with a debugging workflow.
